# Assay: GPU health check on Colab

Runs `assay check` on the free GPU Colab lends you, signs the report, verifies it, and downloads it.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

Takes about 2–3 minutes (most of it installing Go and building). Read-only: Assay never changes any GPU setting.

## 1. Which GPU did we get?
Colab often gives you a different physical GPU each session. The serial number tells them apart.

In [ ]:
!nvidia-smi --query-gpu=name,serial,uuid,driver_version,temperature.gpu --format=csv

## 2. Settings
`BRANCH` is the branch of the repo to build. Use `main` once your work is merged; until then, put your feature branch here (e.g. `day-5`).

In [ ]:
REPO = "https://github.com/anushka11p/anushka11p-Assay-health-grades-for-GPU-backed-debt.git"
BRANCH = "main"
GO_VERSION = "1.27.3"   # must be >= the version in agent/go.mod

## 3. Install Go (skipped if already installed this session)

In [ ]:
%%bash -s "$GO_VERSION"
set -euo pipefail
if ! /usr/local/go/bin/go version 2>/dev/null | grep -q "go$1 "; then
  curl -fsSL "https://go.dev/dl/go$1.linux-amd64.tar.gz" -o /tmp/go.tgz
  rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tgz
fi
/usr/local/go/bin/go version

## 4. Download and build Assay

In [ ]:
%%bash -s "$REPO" "$BRANCH"
set -euo pipefail
rm -rf /content/assay-src
git clone --quiet --depth 1 --branch "$2" "$1" /content/assay-src
cd /content/assay-src/agent
# NVIDIA's C headers print many harmless "deprecated" warnings; show only real errors.
if ! /usr/local/go/bin/go build -o /content/assay ./cmd/assay 2>/tmp/build.log; then
  grep -v -e deprecated -e nvml.h -e cgo-gcc-prolog /tmp/build.log
  exit 1
fi
/content/assay version

## 5. Create a signing key for this session
Colab wipes everything when the session ends, so each session gets a fresh key. The public key is saved next to the report so anyone can verify it.

In [ ]:
!test -f /content/.assay/agent.key || /content/assay keygen --dir /content/.assay

## 6. Run the health check
Reads every GPU counter, then takes 20 readings 250 ms apart (5 seconds). The GPU is idle, so expect low clocks and `gpu_idle`: the stress tests that put it under load come in Week 2.

In [ ]:
import datetime, subprocess
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REPORT = f"/content/report-colab-{stamp}.json"
!/content/assay check --backend=nvml --key /content/.assay/agent.key --out {REPORT}

## 7. Verify the signature
Pinned to this session's public key, so it checks both that the report is unmodified and who signed it.

In [ ]:
!/content/assay verify --pubkey /content/.assay/agent.pub {REPORT}

## 8. Download the report and public key
Put both in `census/raw/` in your repo. Keep the `.pub` with its report: it's what verifiers pin.

In [ ]:
import shutil
PUB = REPORT.replace(".json", ".pub")
shutil.copy("/content/.assay/agent.pub", PUB)
from google.colab import files
files.download(REPORT)
files.download(PUB)